In [2]:
#in here we will install and import dependencies
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install easyocr opencv-python pandas tqdm

import os, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

import cv2
import easyocr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 22.6 MB/s eta 0:00:00


In [3]:
#in here we will mount Google Drive and define dataset paths
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
#in here we will mount Google Drive and define dataset paths
from google.colab import drive
drive.mount('/content/drive')

# CHANGE THIS to your actual folder on Drive
DATA_ROOT = Path("/content/drive/MyDrive/layer1")

# Expected structure:
# layer1/egyptian/train, layer1/egyptian/test
# layer1/foreign/train,  layer1/foreign/test
train_dir = DATA_ROOT / "train"
test_dir  = DATA_ROOT / "test"

# If your current structure is layer1/egyptian/train etc., we will auto-build train/test symlinks-like loaders.
# We'll detect your structure and set train_dir/test_dir accordingly.
if (DATA_ROOT / "egyptian" / "train").exists():
    train_dir = DATA_ROOT  # we'll handle nested class/train in a custom dataset wrapper below
    test_dir  = DATA_ROOT
print("DATA_ROOT:", DATA_ROOT)
print("train_dir:", train_dir)
print("test_dir :", test_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_ROOT: /content/drive/MyDrive/layer1
train_dir: /content/drive/MyDrive/layer1
test_dir : /content/drive/MyDrive/layer1


In [ ]:
#in here we will build datasets and dataloaders (with a validation split from train)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2  # increase to 4 if Colab is stable

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomApply([transforms.ColorJitter(0.2,0.2,0.2,0.1)], p=0.5),
    transforms.RandomRotation(5),
    transforms.RandomHorizontalFlip(p=0.05),  # IDs shouldn't flip often; keep small
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

test_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class NestedTrainTestImageFolder(torch.utils.data.Dataset):
    """
    Supports your structure:
      layer1/egyptian/train/*.jpg
      layer1/egyptian/test/*.jpg
      layer1/foreign/train/*.jpg
      layer1/foreign/test/*.jpg
    Returns an ImageFolder-like dataset for a given split.
    """
    def __init__(self, root: Path, split: str, transform=None):
        assert split in ["train", "test"]
        self.transform = transform
        self.samples = []
        self.class_to_idx = {"egyptian": 0, "foreign": 1}  # keep stable for downstream
        for cls in ["egyptian", "foreign"]:
            p = root / cls / split
            if not p.exists():
                raise FileNotFoundError(f"Missing folder: {p}")
            for ext in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp"):
                for fp in p.glob(ext):
                    self.samples.append((str(fp), self.class_to_idx[cls]))
        random.shuffle(self.samples)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        if img is None:
            raise RuntimeError(f"Failed reading image: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # convert to PIL-like tensor via torchvision transforms by going through numpy->PIL
        # easiest: use transforms.functional.to_pil_image
        import torchvision.transforms.functional as F
        pil = F.to_pil_image(img)
        if self.transform: pil = self.transform(pil)
        return pil, label, path  # include path for reporting

# Choose loader based on detected structure
use_nested = (DATA_ROOT / "egyptian" / "train").exists()

if use_nested:
    full_train = NestedTrainTestImageFolder(DATA_ROOT, "train", transform=train_tfms)
    test_ds    = NestedTrainTestImageFolder(DATA_ROOT, "test",  transform=test_tfms)
    class_names = ["egyptian", "foreign"]
else:
    # If you already have ImageFolder format: layer1/train/egyptian, layer1/train/foreign
    full_train = datasets.ImageFolder(train_dir, transform=train_tfms)
    test_ds    = datasets.ImageFolder(test_dir,  transform=test_tfms)
    class_names = full_train.classes

# validation split
val_ratio = 0.15
val_size = int(len(full_train) * val_ratio)
train_size = len(full_train) - val_size
train_ds, val_ds = random_split(full_train, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True)

train_loader = make_loader(train_ds, shuffle=True)
val_loader   = make_loader(val_ds, shuffle=False)
test_loader  = make_loader(test_ds, shuffle=False)

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))
print("Classes:", class_names)


In [6]:
#in here we will compute class weights to handle imbalance
# We compute weights from the *original training dataset* (before val split) for stability.
if use_nested:
    labels = [y for _, y in full_train.samples]
else:
    labels = [y for _, y in full_train.samples]

counts = np.bincount(labels, minlength=2).astype(np.float32)
weights = counts.sum() / (2.0 * counts + 1e-8)  # inverse frequency
class_weights = torch.tensor(weights, dtype=torch.float32)

print("Class counts (train):", counts, "=> weights:", class_weights.tolist())


Class counts (train): [664. 466.] => weights: [0.8509036302566528, 1.2124463319778442]


In [7]:
#in here we will define the 3 candidate models with a shared training wrapper
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

def build_model(name: str, num_classes=2):
    name = name.lower()
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        input_size = 224
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        input_size = 224
    elif name == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
        input_size = 224
    else:
        raise ValueError("Unknown model name. Use: resnet18, efficientnet_b0, mobilenet_v3_small")
    return m.to(DEVICE), input_size

def accuracy_from_logits(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for batch in loader:
        if use_nested:
            x, y, _paths = batch
        else:
            x, y = batch
        x = x.to(DEVICE); y = y.to(DEVICE)
        logits = model(x)
        if criterion is not None:
            loss = criterion(logits, y).item()
            total_loss += loss * x.size(0)
        total_acc += (torch.argmax(logits,1) == y).sum().item()
        n += x.size(0)
    avg_loss = total_loss / max(n,1) if criterion is not None else None
    avg_acc = total_acc / max(n,1)
    return avg_loss, avg_acc

def train_one_model(model_name, epochs=6, lr=3e-4):
    model, _ = build_model(model_name, num_classes=2)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc = -1
    best_state = None
    history = []

    for ep in range(1, epochs+1):
        model.train()
        running_loss, running_acc, n = 0.0, 0.0, 0
        for batch in train_loader:
            if use_nested:
                x, y, _paths = batch
            else:
                x, y = batch
            x = x.to(DEVICE); y = y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            running_acc  += (torch.argmax(logits,1) == y).sum().item()
            n += x.size(0)

        scheduler.step()
        train_loss = running_loss / n
        train_acc  = running_acc / n

        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history.append({"epoch": ep, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc})

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}

        print(f"{model_name} | epoch {ep}/{epochs} | train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

    # load best
    model.load_state_dict(best_state)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    return model, pd.DataFrame(history), {"model": model_name, "best_val_acc": best_val_acc, "test_acc": test_acc, "test_loss": test_loss}


DEVICE: cpu


In [8]:
#in here we will train all 3 models and compare their results
EPOCHS = 6  # increase to 10-15 once pipeline works
LR = 3e-4

results = []
histories = {}
trained_models = {}

for mname in ["efficientnet_b0", "mobilenet_v3_small", "resnet18"]:
    model, hist_df, summary = train_one_model(mname, epochs=EPOCHS, lr=LR)
    results.append(summary)
    histories[mname] = hist_df
    trained_models[mname] = model

results_df = pd.DataFrame(results).sort_values("test_acc", ascending=False).reset_index(drop=True)
results_df


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 59.2MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


efficientnet_b0 | epoch 1/6 | train_acc=0.9802 val_acc=1.0000
efficientnet_b0 | epoch 2/6 | train_acc=0.9990 val_acc=1.0000
efficientnet_b0 | epoch 3/6 | train_acc=1.0000 val_acc=1.0000
efficientnet_b0 | epoch 4/6 | train_acc=0.9990 val_acc=1.0000
efficientnet_b0 | epoch 5/6 | train_acc=0.9990 val_acc=1.0000
efficientnet_b0 | epoch 6/6 | train_acc=1.0000 val_acc=1.0000
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 47.8MB/s]


mobilenet_v3_small | epoch 1/6 | train_acc=0.9750 val_acc=0.7633
mobilenet_v3_small | epoch 2/6 | train_acc=0.9990 val_acc=1.0000
mobilenet_v3_small | epoch 3/6 | train_acc=0.9990 val_acc=1.0000
mobilenet_v3_small | epoch 4/6 | train_acc=0.9990 val_acc=1.0000
mobilenet_v3_small | epoch 5/6 | train_acc=1.0000 val_acc=1.0000
mobilenet_v3_small | epoch 6/6 | train_acc=1.0000 val_acc=1.0000
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 178MB/s]


resnet18 | epoch 1/6 | train_acc=0.9844 val_acc=1.0000
resnet18 | epoch 2/6 | train_acc=0.9969 val_acc=0.9941
resnet18 | epoch 3/6 | train_acc=0.9958 val_acc=1.0000
resnet18 | epoch 4/6 | train_acc=0.9969 val_acc=1.0000
resnet18 | epoch 5/6 | train_acc=1.0000 val_acc=1.0000
resnet18 | epoch 6/6 | train_acc=1.0000 val_acc=1.0000


,model,best_val_acc,test_acc,test_loss
0,efficientnet_b0,1.0,1.0,0.000598
1,mobilenet_v3_small,1.0,1.0,0.009810
2,resnet18,1.0,1.0,0.001619


In [ ]:
#in here we will select the best model, save it to Drive, and create a clean inference function for Layer 2
BEST_MODEL_NAME = results_df.loc[0, "model"]
best_model = trained_models[BEST_MODEL_NAME].eval()

SAVE_DIR = Path("/content/drive/MyDrive/egyid_layer1_artifacts")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

best_path = SAVE_DIR / f"layer1_best_{BEST_MODEL_NAME}.pt"
torch.save({
    "model_name": BEST_MODEL_NAME,
    "state_dict": best_model.state_dict(),
    "class_to_idx": {"egyptian": 0, "foreign": 1},
    "img_size": IMG_SIZE
}, best_path)

print("Saved best model to:", best_path)

# inference transform (match test)
infer_tfms = test_tfms

@torch.no_grad()
def predict_egyptian_or_foreign(pil_image):
    x = infer_tfms(pil_image).unsqueeze(0).to(DEVICE)
    logits = best_model(x)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred = int(np.argmax(probs))
    label = "egyptian" if pred == 0 else "foreign"
    conf = float(probs[pred])
    return label, conf, probs.tolist()


In [ ]:
#in here we will run inference on the test set and generate a report (confusion matrix + misclassified samples)
from sklearn.metrics import confusion_matrix, classification_report

y_true, y_pred, rows = [], [], []

best_model.eval()
for batch in test_loader:
    if use_nested:
        x, y, paths = batch
    else:
        x, y = batch
        paths = ["(path unavailable in ImageFolder default)"] * len(y)
    x = x.to(DEVICE)
    logits = best_model(x)
    preds = torch.argmax(logits, dim=1).cpu().numpy().tolist()
    y_true.extend(y.numpy().tolist() if hasattr(y, "numpy") else y.tolist())
    y_pred.extend(preds)
    for pth, yt, yp in zip(paths, y, preds):
        rows.append({"path": pth, "true": int(yt), "pred": int(yp)})

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix (rows=true, cols=pred):\n", cm)
print("\nReport:\n", classification_report(y_true, y_pred, target_names=["egyptian","foreign"]))

rep_df = pd.DataFrame(rows)
mis_df = rep_df[rep_df.true != rep_df.pred].copy()
mis_path = SAVE_DIR / f"layer1_misclassified_{BEST_MODEL_NAME}.csv"
mis_df.to_csv(mis_path, index=False)
print("Saved misclassifications to:", mis_path)


In [ ]:
#in here we will save Layer 1 results (metrics table) to Drive
from pathlib import Path
import pandas as pd

SAVE_DIR = Path("/content/drive/MyDrive/egyid_layer1_artifacts")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(SAVE_DIR / "layer1_model_comparison.csv", index=False)
print("Saved:", SAVE_DIR / "layer1_model_comparison.csv")


In [ ]:
#in here we will check for duplicate images between train and test (sanity check)
import hashlib
from pathlib import Path
from tqdm import tqdm

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def collect_hashes(folder: Path):
    files = []
    for ext in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp"):
        files += list(folder.rglob(ext))
    hashes = {}
    for fp in tqdm(files, desc=str(folder.name)):
        hashes[str(fp)] = md5(fp)
    return set(hashes.values())

# update this if your structure differs
train_hashes = collect_hashes(Path("/content/drive/MyDrive/layer1/egyptian/train")) | collect_hashes(Path("/content/drive/MyDrive/layer1/foreign/train"))
test_hashes  = collect_hashes(Path("/content/drive/MyDrive/layer1/egyptian/test"))  | collect_hashes(Path("/content/drive/MyDrive/layer1/foreign/test"))

overlap = train_hashes.intersection(test_hashes)
print("Duplicate hash overlap count:", len(overlap))


In [13]:
#in here we will list the exact duplicate files between train and test
import hashlib
from pathlib import Path
from tqdm import tqdm

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def hash_to_paths(folder: Path):
    files = []
    for ext in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp"):
        files += list(folder.rglob(ext))
    mapping = {}
    for fp in tqdm(files, desc=str(folder.name)):
        hv = md5(fp)
        mapping.setdefault(hv, []).append(fp)
    return mapping

ROOT = Path("/content/drive/MyDrive/layer1")  # change if needed

train_map = {}
for cls in ["egyptian", "foreign"]:
    train_map.update(hash_to_paths(ROOT / cls / "train"))

test_map = {}
for cls in ["egyptian", "foreign"]:
    test_map.update(hash_to_paths(ROOT / cls / "test"))

train_hashes = set(train_map.keys())
test_hashes  = set(test_map.keys())
overlap = sorted(list(train_hashes & test_hashes))

print("Overlap hashes:", len(overlap))
for hv in overlap[:20]:
    print("\nHASH:", hv)
    print("  TRAIN:", [str(p) for p in train_map[hv]])
    print("  TEST :", [str(p) for p in test_map[hv]])


test: 100%|██████████| 53/53 [00:00<00:00, 163.46it/s]

Overlap hashes: 0


In [ ]:
#in here we will move duplicated test images out of the test set (safe cleanup)
import shutil
from pathlib import Path

DUP_DIR = ROOT / "_duplicates_removed"
DUP_DIR.mkdir(exist_ok=True)

moved = 0
for hv in overlap:
    for test_fp in test_map[hv]:
        # move the test duplicate away
        dest = DUP_DIR / test_fp.name
        shutil.move(str(test_fp), str(dest))
        moved += 1

print(f"Moved {moved} duplicate test files to: {DUP_DIR}")


In [ ]:
#in here we will rebuild the test loader and re-evaluate the trained best model without retraining
from torch.utils.data import DataLoader

# rebuild test dataset (same transforms)
if use_nested:
    test_ds_clean = NestedTrainTestImageFolder(ROOT, "test", transform=test_tfms)
else:
    test_ds_clean = datasets.ImageFolder(test_dir, transform=test_tfms)

test_loader_clean = DataLoader(test_ds_clean, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
test_loss, test_acc = evaluate(best_model, test_loader_clean, criterion)
print("CLEAN test size:", len(test_ds_clean))
print("CLEAN test acc :", test_acc)
print("CLEAN test loss:", test_loss)


In [ ]:
#in here we will save a Layer 1 lock metadata file for reproducibility
from pathlib import Path
import json, time

SAVE_DIR = Path("/content/drive/MyDrive/egyid_layer1_artifacts")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

layer1_lock = {
    "layer": "layer1_gate_egyptian_vs_foreign",
    "locked_at_unix": int(time.time()),
    "best_model_ckpt": str(SAVE_DIR / "layer1_best_efficientnet_b0.pt"),
    "best_model_name": BEST_MODEL_NAME if "BEST_MODEL_NAME" in globals() else "efficientnet_b0",
    "img_size": IMG_SIZE if "IMG_SIZE" in globals() else 224,
    "classes": ["egyptian", "foreign"],
    "class_to_idx": {"egyptian": 0, "foreign": 1},
    "clean_test_size": 109,
    "clean_test_acc": 1.0,
    "notes": [
        "Removed 3 exact duplicate foreign images from test set to prevent leakage",
        "Re-evaluated on clean test set; accuracy remained 1.0"
    ]
}

with open(SAVE_DIR / "layer1_lock.json", "w") as f:
    json.dump(layer1_lock, f, indent=2, ensure_ascii=False)

print("Saved:", SAVE_DIR / "layer1_lock.json")


In [ ]:
#in here we will define a clean Layer 1 Gate API to be reused by Layer 2
import torch
import torch.nn as nn
import numpy as np
import cv2
from pathlib import Path
from torchvision import models, transforms
import torchvision.transforms.functional as F

class Layer1Gate:
    def __init__(self, ckpt_path: str, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        ckpt = torch.load(ckpt_path, map_location="cpu")

        self.model_name = ckpt["model_name"]
        self.img_size = ckpt.get("img_size", 224)
        self.class_to_idx = ckpt["class_to_idx"]
        self.idx_to_class = {v:k for k,v in self.class_to_idx.items()}

        self.model = self._build_model(self.model_name, num_classes=2)
        self.model.load_state_dict(ckpt["state_dict"])
        self.model = self.model.to(self.device).eval()

        self.tfms = transforms.Compose([
            transforms.Resize((self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ])

    def _build_model(self, name: str, num_classes=2):
        name = name.lower()
        if name == "resnet18":
            m = models.resnet18(weights=None)
            m.fc = nn.Linear(m.fc.in_features, num_classes)
        elif name == "efficientnet_b0":
            m = models.efficientnet_b0(weights=None)
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        elif name == "mobilenet_v3_small":
            m = models.mobilenet_v3_small(weights=None)
            m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
        else:
            raise ValueError(f"Unknown model name in checkpoint: {name}")
        return m

    @torch.no_grad()
    def predict(self, image_path: str):
        bgr = cv2.imread(image_path)
        if bgr is None:
            return {"status":"error", "reason": f"cannot read image: {image_path}"}
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        pil = F.to_pil_image(rgb)

        x = self.tfms(pil).unsqueeze(0).to(self.device)
        logits = self.model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

        pred_idx = int(np.argmax(probs))
        pred_label = self.idx_to_class[pred_idx]
        conf = float(probs[pred_idx])

        return {
            "status": "ok",
            "pred": pred_label,          # "egyptian" or "foreign"
            "conf": conf,
            "probs": probs.tolist(),
            "image_path": image_path
        }

CKPT_PATH = "/content/drive/MyDrive/egyid_layer1_artifacts/layer1_best_efficientnet_b0.pt"
layer1_gate = Layer1Gate(CKPT_PATH)
print("Layer1Gate loaded:", layer1_gate.model_name, "| device:", layer1_gate.device)


In [18]:
#in here we will run Layer 1 on a folder and save a manifest for Layer 2
import pandas as pd
from tqdm import tqdm
from pathlib import Path

INPUT_FOLDER = Path("/content/drive/MyDrive/layer1/egyptian/test")  # change to your real incoming folder
OUT_DIR = Path("/content/drive/MyDrive/egyid_layer1_artifacts/layer1_runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

img_paths = []
for ext in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp"):
    img_paths += list(INPUT_FOLDER.rglob(ext))
img_paths = sorted(img_paths)

rows = []
for p in tqdm(img_paths):
    rows.append(layer1_gate.predict(str(p)))

df = pd.DataFrame(rows)
run_path = OUT_DIR / "layer1_run_manifest.csv"
df.to_csv(run_path, index=False)
print("Saved run manifest:", run_path)
df.head()


100%|██████████| 56/56 [00:03<00:00, 14.33it/s]


Saved run manifest: /content/drive/MyDrive/egyid_layer1_artifacts/layer1_runs/layer1_run_manifest.csv


,status,pred,conf,probs,image_path
0,ok,egyptian,0.999106,"[0.9991059899330139, 0.0008940442930907011]",/content/drive/MyDrive/layer1/egyptian/test/ID...
1,ok,egyptian,0.999417,"[0.9994169473648071, 0.0005830308655276895]",/content/drive/MyDrive/layer1/egyptian/test/ID...
2,ok,egyptian,0.999181,"[0.9991808533668518, 0.0008190926164388657]",/content/drive/MyDrive/layer1/egyptian/test/ID...
3,ok,egyptian,0.999372,"[0.9993717074394226, 0.0006283395341597497]",/content/drive/MyDrive/layer1/egyptian/test/ID...
4,ok,egyptian,0.999048,"[0.9990478157997131, 0.0009521772153675556]",/content/drive/MyDrive/layer1/egyptian/test/ID...


In [ ]:
#in here we will mount Drive and define Layer 2 dataset paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
LAYER2_ROOT = Path("/content/drive/MyDrive/layer2")

print("Layer2 root exists:", LAYER2_ROOT.exists())
for p in [LAYER2_ROOT/"front", LAYER2_ROOT/"back"]:
    print(p, "exists:", p.exists())


In [ ]:
#in here we will define transforms and a dataset class for layer2/front|back with train/valid/test splits
import random, numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
import cv2
import torchvision.transforms.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomApply([transforms.ColorJitter(0.2,0.2,0.2,0.1)], p=0.5),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class Layer2SplitDataset(torch.utils.data.Dataset):
    """
    Expects:
      root/front/train/*.jpg
      root/front/valid/*.jpg
      root/front/test/*.jpg
      root/back/train/*.jpg
      root/back/valid/*.jpg
      root/back/test/*.jpg
    """
    def __init__(self, root: Path, split: str, transform=None):
        assert split in ["train", "valid", "test"]
        self.root = root
        self.split = split
        self.transform = transform

        self.class_to_idx = {"front": 0, "back": 1}  # stable mapping for pipeline
        self.samples = []

        for cls in ["front", "back"]:
            folder = root / cls / split
            if not folder.exists():
                raise FileNotFoundError(f"Missing folder: {folder}")
            for ext in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp"):
                for fp in folder.glob(ext):
                    self.samples.append((str(fp), self.class_to_idx[cls]))

        random.shuffle(self.samples)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        bgr = cv2.imread(path)
        if bgr is None:
            raise RuntimeError(f"Failed reading image: {path}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        pil = F.to_pil_image(rgb)
        if self.transform:
            pil = self.transform(pil)
        return pil, label, path

train_ds = Layer2SplitDataset(LAYER2_ROOT, "train", transform=train_tfms)
val_ds   = Layer2SplitDataset(LAYER2_ROOT, "valid", transform=eval_tfms)
test_ds  = Layer2SplitDataset(LAYER2_ROOT, "test",  transform=eval_tfms)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

train_loader = make_loader(train_ds, shuffle=True)
val_loader   = make_loader(val_ds, shuffle=False)
test_loader  = make_loader(test_ds, shuffle=False)

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))
print("class_to_idx:", train_ds.class_to_idx)


In [21]:
#in here we will compute class weights to handle imbalance for front/back
labels = [y for _, y in train_ds.samples]
counts = np.bincount(labels, minlength=2).astype(np.float32)
weights = counts.sum() / (2.0 * counts + 1e-8)
class_weights = torch.tensor(weights, dtype=torch.float32)

print("Counts [front, back]:", counts)
print("Weights:", class_weights.tolist())


Counts [front, back]: [1569.  975.]
Weights: [0.8107074499130249, 1.3046153783798218]


In [ ]:
#in here we will define training and evaluation functions for 3 candidate models
from tqdm import tqdm
import pandas as pd

def build_model(name: str, num_classes=2):
    name = name.lower()
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif name == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    else:
        raise ValueError("Use: resnet18, efficientnet_b0, mobilenet_v3_small")
    return m.to(DEVICE)

@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y, _paths in loader:
        x = x.to(DEVICE); y = y.to(DEVICE)
        logits = model(x)
        if criterion is not None:
            loss = criterion(logits, y).item()
            total_loss += loss * x.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        n += x.size(0)
    avg_loss = total_loss / max(n,1) if criterion is not None else None
    avg_acc = correct / max(n,1)
    return avg_loss, avg_acc

def train_one_model(model_name, epochs=6, lr=3e-4):
    model = build_model(model_name, num_classes=2)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc = -1
    best_state = None
    history = []

    for ep in range(1, epochs+1):
        model.train()
        running_loss, correct, n = 0.0, 0, 0

        for x, y, _paths in train_loader:
            x = x.to(DEVICE); y = y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            correct += (torch.argmax(logits,1) == y).sum().item()
            n += x.size(0)

        scheduler.step()
        train_loss = running_loss / n
        train_acc = correct / n

        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history.append({"epoch": ep, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc})

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}

        print(f"{model_name} | epoch {ep}/{epochs} | train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

    model.load_state_dict(best_state)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    return model, pd.DataFrame(history), {"model": model_name, "best_val_acc": best_val_acc, "test_acc": test_acc, "test_loss": test_loss}


In [ ]:
#in here we will train all 3 models and compare their results
EPOCHS = 6
LR = 3e-4

results = []
histories = {}
trained_models = {}

for mname in ["efficientnet_b0", "resnet18", "mobilenet_v3_small"]:
    model, hist_df, summary = train_one_model(mname, epochs=EPOCHS, lr=LR)
    results.append(summary)
    histories[mname] = hist_df
    trained_models[mname] = model

results_df = pd.DataFrame(results).sort_values("test_acc", ascending=False).reset_index(drop=True)
results_df


In [ ]:
#in here we will check for duplicate images across train/valid/test for Layer 2
import hashlib
from pathlib import Path
from tqdm import tqdm

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def collect_hashes(folder: Path):
    files = []
    for ext in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp"):
        files += list(folder.rglob(ext))
    hashes = {}
    for fp in tqdm(files, desc=str(folder)):
        hashes[str(fp)] = md5(fp)
    return hashes

ROOT = Path("/content/drive/MyDrive/layer2")  # change if needed

splits = ["train", "valid", "test"]
hashes = {s: {} for s in splits}

for s in splits:
    for cls in ["front", "back"]:
        folder = ROOT / cls / s
        hashes[s].update(collect_hashes(folder))

set_train = set(hashes["train"].values())
set_valid = set(hashes["valid"].values())
set_test  = set(hashes["test"].values())

print("Duplicates train∩valid:", len(set_train & set_valid))
print("Duplicates train∩test :", len(set_train & set_test))
print("Duplicates valid∩test :", len(set_valid & set_test))


In [ ]:
#in here we will choose which Layer 2 model to lock (tie-breaker for pipeline speed)
# If accuracy is tied, prefer MobileNet for speed in the full pipeline.
PREFERRED = "mobilenet_v3_small"

if (results_df["model"] == PREFERRED).any() and float(results_df.loc[results_df["model"]==PREFERRED, "test_acc"].values[0]) == float(results_df["test_acc"].max()):
    BEST2_MODEL_NAME = PREFERRED
else:
    BEST2_MODEL_NAME = results_df.loc[0, "model"]

best2_model = trained_models[BEST2_MODEL_NAME].eval()
print("Chosen Layer 2 model to lock:", BEST2_MODEL_NAME)


In [ ]:
#in here we will save the locked Layer 2 artifacts to Drive
import json, time
from pathlib import Path
import torch

SAVE2_DIR = Path("/content/drive/MyDrive/egyid_layer2_artifacts")
SAVE2_DIR.mkdir(parents=True, exist_ok=True)

ckpt_path = SAVE2_DIR / f"layer2_best_{BEST2_MODEL_NAME}.pt"
torch.save({
    "model_name": BEST2_MODEL_NAME,
    "state_dict": best2_model.state_dict(),
    "class_to_idx": {"front": 0, "back": 1},
    "img_size": IMG_SIZE
}, ckpt_path)

results_df.to_csv(SAVE2_DIR / "layer2_model_comparison.csv", index=False)

layer2_lock = {
    "layer": "layer2_side_front_vs_back",
    "locked_at_unix": int(time.time()),
    "best_model_ckpt": str(ckpt_path),
    "best_model_name": BEST2_MODEL_NAME,
    "img_size": IMG_SIZE,
    "classes": ["front", "back"],
    "class_to_idx": {"front": 0, "back": 1},
    "test_acc": float(results_df.loc[results_df["model"]==BEST2_MODEL_NAME, "test_acc"].values[0]),
    "notes": [
        "Used provided valid/ split (no re-splitting).",
        "Used class-weighted loss due to imbalance.",
        "Tie-breaker: prefer smaller/faster model for pipeline (MobileNet) when accuracy matches."
    ]
}

with open(SAVE2_DIR / "layer2_lock.json", "w") as f:
    json.dump(layer2_lock, f, indent=2)

print("Saved best ckpt:", ckpt_path)
print("Saved:", SAVE2_DIR / "layer2_model_comparison.csv")
print("Saved:", SAVE2_DIR / "layer2_lock.json")


In [ ]:
#in here we will create the Layer1->Layer2 combined manifest for Layer 3
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
import numpy as np
import cv2
import torchvision.transforms.functional as F
from torchvision import transforms
import torch.nn as nn
from torchvision import models

LAYER1_MANIFEST = Path("/content/drive/MyDrive/egyid_layer1_artifacts/layer1_runs/layer1_run_manifest.csv")

# Load Layer 1 manifest
df1 = pd.read_csv(LAYER1_MANIFEST)
df_egy = df1[(df1["status"]=="ok") & (df1["pred"]=="egyptian")].copy()

# Load Layer 2 checkpoint and build predictor quickly
ckpt = torch.load(str(SAVE2_DIR / f"layer2_best_{BEST2_MODEL_NAME}.pt"), map_location="cpu")
model_name = ckpt["model_name"]
img_size = ckpt.get("img_size", 224)
class_to_idx = ckpt["class_to_idx"]
idx_to_class = {v:k for k,v in class_to_idx.items()}

def build_model_infer(name: str):
    name = name.lower()
    if name == "resnet18":
        m = models.resnet18(weights=None); m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None); m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    elif name == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights=None); m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    else:
        raise ValueError("Unknown")
    return m

layer2_model = build_model_infer(model_name)
layer2_model.load_state_dict(ckpt["state_dict"])
layer2_model = layer2_model.to(DEVICE).eval()

tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

@torch.no_grad()
def layer2_predict(image_path: str):
    bgr = cv2.imread(image_path)
    if bgr is None:
        return {"status":"error", "reason":"cannot read", "image_path": image_path}
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    pil = F.to_pil_image(rgb)
    x = tfms(pil).unsqueeze(0).to(DEVICE)
    probs = torch.softmax(layer2_model(x), dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    return {"status":"ok", "side": idx_to_class[pred_idx], "conf": float(probs[pred_idx]), "probs": probs.tolist(), "image_path": image_path}

rows = []
for p in tqdm(df_egy["image_path"].tolist()):
    rows.append(layer2_predict(p))

df2 = pd.DataFrame(rows)
df_out = df_egy.merge(df2, on="image_path", how="left", suffixes=("_layer1","_layer2"))

OUT_PATH = Path("/content/drive/MyDrive/egyid_layer2_artifacts/layer2_from_layer1_manifest.csv")
df_out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
df_out.head()


In [ ]:
#in here we will list the exact duplicate files between train and valid in Layer 2
import hashlib
from pathlib import Path
from tqdm import tqdm

ROOT = Path("/content/drive/MyDrive/layer2")  # change if needed

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def hash_map(folder: Path):
    files = []
    for ext in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp"):
        files += list(folder.rglob(ext))
    m = {}
    for fp in tqdm(files, desc=str(folder)):
        hv = md5(fp)
        m.setdefault(hv, []).append(fp)
    return m

train_map = {}
valid_map = {}

for cls in ["front", "back"]:
    train_map.update(hash_map(ROOT/cls/"train"))
    valid_map.update(hash_map(ROOT/cls/"valid"))

overlap = sorted(list(set(train_map.keys()) & set(valid_map.keys())))
print("Train∩Valid duplicate hashes:", len(overlap))

for hv in overlap:
    print("\nHASH:", hv)
    print("  TRAIN:", [str(p) for p in train_map[hv]])
    print("  VALID:", [str(p) for p in valid_map[hv]])


In [ ]:
#in here we will move duplicated valid images out of the valid set (safe cleanup)
import shutil

DUP_DIR = ROOT / "_duplicates_removed_layer2_valid"
DUP_DIR.mkdir(exist_ok=True)

moved = 0
for hv in overlap:
    for fp in valid_map[hv]:
        dest = DUP_DIR / fp.name
        shutil.move(str(fp), str(dest))
        moved += 1

print(f"Moved {moved} duplicated VALID files to: {DUP_DIR}")


In [ ]:
#in here we will rebuild the valid loader and re-evaluate the locked model after cleanup
from torch.utils.data import DataLoader

val_ds_clean = Layer2SplitDataset(LAYER2_ROOT, "valid", transform=eval_tfms)
val_loader_clean = DataLoader(val_ds_clean, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
val_loss, val_acc = evaluate(best2_model, val_loader_clean, criterion)

print("CLEAN valid size:", len(val_ds_clean))
print("CLEAN valid acc :", val_acc)
print("CLEAN valid loss:", val_loss)


In [ ]:
#in here we will set Layer 3 input/output paths and create output folders
from pathlib import Path
import pandas as pd

LAYER2_MANIFEST = Path("/content/drive/MyDrive/egyid_layer2_artifacts/layer2_from_layer1_manifest.csv")

LAYER3_DIR = Path("/content/drive/MyDrive/egyid_layer3_artifacts")
UPRIGHT_DIR = LAYER3_DIR / "upright_images"
LAYER3_DIR.mkdir(parents=True, exist_ok=True)
UPRIGHT_DIR.mkdir(parents=True, exist_ok=True)

print("Layer2 manifest:", LAYER2_MANIFEST, "exists:", LAYER2_MANIFEST.exists())
print("Layer3 dir:", LAYER3_DIR)
print("Upright dir:", UPRIGHT_DIR)

In [ ]:
#in here we will load Layer 2 manifest and filter only Egyptian IDs with valid side predictions
df = pd.read_csv(LAYER2_MANIFEST)

# Expecting columns: pred, status_layer2, side, image_path (based on your merge)
df_egy = df[
    (df["pred"] == "egyptian") &
    (df["status_layer2"] == "ok") &
    (df["side"].isin(["front", "back"]))
].copy()

print("Total rows in manifest:", len(df))
print("Egyptian rows for Layer 3:", len(df_egy))
df_egy.head()

In [ ]:
#in here we will initialize OCR engine used to score best rotation (rotation only)
!pip -q install easyocr opencv-python

import cv2
import numpy as np
import torch
from tqdm import tqdm
import easyocr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# Arabic+English is useful because some IDs include Latin numbers/characters too
reader = easyocr.Reader(['ar','en'], gpu=torch.cuda.is_available())

In [ ]:
#in here we will define rotation candidates and an OCR-based scoring function
def rotate_rgb(img_rgb, angle):
    if angle == 0:
        return img_rgb
    if angle == 90:
        return cv2.rotate(img_rgb, cv2.ROTATE_90_CLOCKWISE)
    if angle == 180:
        return cv2.rotate(img_rgb, cv2.ROTATE_180)
    if angle == 270:
        return cv2.rotate(img_rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)
    raise ValueError("angle must be 0/90/180/270")

def ocr_score(img_rgb):
    """
    Returns:
      score: combined score
      mean_conf: avg OCR confidence
      digit_count: count of digits detected (Arabic or Latin)
      text_len: total length of detected text
    """
    try:
        results = reader.readtext(img_rgb, detail=1)
    except Exception:
        return -1e9, 0.0, 0, 0

    if not results:
        return -1e9, 0.0, 0, 0

    confs = [r[2] for r in results if isinstance(r[2], (float,int))]
    mean_conf = float(np.mean(confs)) if confs else 0.0

    text_all = " ".join([r[1] for r in results if isinstance(r[1], str)])
    digit_count = sum(ch.isdigit() for ch in text_all)
    text_len = len(text_all)

    # Combined score: prioritize OCR confidence, then digits/text as signal
    score = mean_conf * 10.0 + min(digit_count, 80) * 0.08 + min(text_len, 160) * 0.01
    return score, mean_conf, digit_count, text_len

def best_rotation_for_image(image_path):
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        return {"status":"error", "reason":"cannot_read", "angle":None, "best_rgb":None,
                "score":None, "mean_conf":None, "digit_count":None, "text_len":None}

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    best = {"angle": 0, "score": -1e9, "mean_conf": 0.0, "digit_count": 0, "text_len": 0, "best_rgb": rgb}
    for angle in [0, 90, 180, 270]:
        rot = rotate_rgb(rgb, angle)
        score, mc, dc, tl = ocr_score(rot)
        if score > best["score"]:
            best.update({"angle": angle, "score": score, "mean_conf": mc, "digit_count": dc, "text_len": tl, "best_rgb": rot})

    best["status"] = "ok"
    return best

In [ ]:
#in here we will define a clean Layer 2 Side API (front/back) from the locked checkpoint
import torch
import torch.nn as nn
import numpy as np
import cv2
from torchvision import models, transforms
import torchvision.transforms.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class Layer2SideClassifier:
    def __init__(self, ckpt_path: str, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        ckpt = torch.load(ckpt_path, map_location="cpu")

        self.model_name = ckpt["model_name"]
        self.img_size = ckpt.get("img_size", 224)
        self.class_to_idx = ckpt["class_to_idx"]
        self.idx_to_class = {v:k for k,v in self.class_to_idx.items()}

        self.model = self._build_model(self.model_name, num_classes=2)
        self.model.load_state_dict(ckpt["state_dict"])
        self.model = self.model.to(self.device).eval()

        self.tfms = transforms.Compose([
            transforms.Resize((self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ])

    def _build_model(self, name: str, num_classes=2):
        name = name.lower()
        if name == "resnet18":
            m = models.resnet18(weights=None)
            m.fc = nn.Linear(m.fc.in_features, num_classes)
        elif name == "efficientnet_b0":
            m = models.efficientnet_b0(weights=None)
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        elif name == "mobilenet_v3_small":
            m = models.mobilenet_v3_small(weights=None)
            m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
        else:
            raise ValueError(f"Unknown model name in checkpoint: {name}")
        return m

    @torch.no_grad()
    def predict(self, image_path: str):
        bgr = cv2.imread(image_path)
        if bgr is None:
            return {"status":"error", "reason": f"cannot read image: {image_path}"}
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        pil = F.to_pil_image(rgb)

        x = self.tfms(pil).unsqueeze(0).to(self.device)
        logits = self.model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

        pred_idx = int(np.argmax(probs))
        pred_label = self.idx_to_class[pred_idx]
        conf = float(probs[pred_idx])

        return {
            "status": "ok",
            "side": pred_label,         # "front" or "back"
            "conf": conf,
            "probs": probs.tolist(),
            "image_path": image_path
        }

LAYER2_CKPT = "/content/drive/MyDrive/egyid_layer2_artifacts/layer2_best_mobilenet_v3_small.pt"
layer2_side = Layer2SideClassifier(LAYER2_CKPT)
print("Layer2SideClassifier loaded:", layer2_side.model_name, "| device:", layer2_side.device)

In [ ]:
#in here we will load the Layer 2 combined manifest and set Layer 3 output paths
from pathlib import Path
import pandas as pd

LAYER2_MANIFEST = Path("/content/drive/MyDrive/egyid_layer2_artifacts/layer2_from_layer1_manifest.csv")
df = pd.read_csv(LAYER2_MANIFEST)

df_egy = df[(df["pred"]=="egyptian") & (df["status_layer2"]=="ok") & (df["side"].isin(["front","back"]))].copy()
print("Images entering Layer 3:", len(df_egy))

LAYER3_DIR = Path("/content/drive/MyDrive/egyid_layer3_artifacts")
UPRIGHT_DIR = LAYER3_DIR / "upright_images"
LAYER3_DIR.mkdir(parents=True, exist_ok=True)
UPRIGHT_DIR.mkdir(parents=True, exist_ok=True)

df_egy.head()

In [ ]:
#in here we will install and initialize OCR used to choose the best rotation
!pip -q install easyocr

import torch
import easyocr

reader = easyocr.Reader(['ar','en'], gpu=torch.cuda.is_available())
print("EasyOCR ready | GPU:", torch.cuda.is_available())

In [ ]:
#in here we will rotate each image (0/90/180/270) and save the best upright version for OCR
import cv2, numpy as np
from tqdm import tqdm

def rotate_rgb(img_rgb, angle):
    if angle == 0:   return img_rgb
    if angle == 90:  return cv2.rotate(img_rgb, cv2.ROTATE_90_CLOCKWISE)
    if angle == 180: return cv2.rotate(img_rgb, cv2.ROTATE_180)
    if angle == 270: return cv2.rotate(img_rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)
    raise ValueError("angle must be 0/90/180/270")

def mean_ocr_conf(img_rgb):
    try:
        res = reader.readtext(img_rgb, detail=1)
    except Exception:
        return -1e9
    if not res:
        return -1e9
    confs = [r[2] for r in res if isinstance(r[2], (float,int))]
    return float(np.mean(confs)) if confs else -1e9

rows = []
for _, r in tqdm(df_egy.iterrows(), total=len(df_egy)):
    p = r["image_path"]
    side = r["side"]

    bgr = cv2.imread(p)
    if bgr is None:
        rows.append({"status_layer3":"error", "image_path":p, "upright_path":None, "side":side, "best_rotation_deg":None, "ocr_score":None})
        continue

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    best_angle, best_score, best_rgb = 0, -1e9, rgb
    for ang in [0,90,180,270]:
        rot = rotate_rgb(rgb, ang)
        sc = mean_ocr_conf(rot)
        if sc > best_score:
            best_angle, best_score, best_rgb = ang, sc, rot

    out_dir = UPRIGHT_DIR / side
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{Path(p).stem}_rot{best_angle}.jpg"
    cv2.imwrite(str(out_path), cv2.cvtColor(best_rgb, cv2.COLOR_RGB2BGR))

    rows.append({
        "status_layer3":"ok",
        "image_path": p,
        "upright_path": str(out_path),
        "side": side,
        "best_rotation_deg": best_angle,
        "ocr_score": best_score,
        "conf_layer1": r.get("conf_layer1", None),
        "conf_layer2": r.get("conf_layer2", None),
    })

layer3_df = pd.DataFrame(rows)
LAYER3_MANIFEST = LAYER3_DIR / "layer3_manifest.csv"
layer3_df.to_csv(LAYER3_MANIFEST, index=False)

print("Saved Layer 3 manifest:", LAYER3_MANIFEST)
print(layer3_df["status_layer3"].value_counts(dropna=False))
layer3_df.head()

-------------

In [ ]:
#in here we will save a Layer 3 lock file for reproducibility and easy resume
import json, time
from pathlib import Path

LAYER3_DIR = Path("/content/drive/MyDrive/egyid_layer3_artifacts")
LOCK_PATH = LAYER3_DIR / "layer3_lock.json"

lock = {
    "layer": "layer3_rotation_normalization",
    "locked_at_unix": int(time.time()),
    "input_manifest": "/content/drive/MyDrive/egyid_layer2_artifacts/layer2_from_layer1_manifest.csv",
    "output_manifest": "/content/drive/MyDrive/egyid_layer3_artifacts/layer3_manifest.csv",
    "upright_dir": "/content/drive/MyDrive/egyid_layer3_artifacts/upright_images",
    "rotations_tested": [0, 90, 180, 270],
    "flip_used": False,
    "crop_used": False,
    "status_counts": layer3_df["status_layer3"].value_counts().to_dict()
}

with open(LOCK_PATH, "w") as f:
    json.dump(lock, f, indent=2)

print("Saved:", LOCK_PATH)

In [ ]:
#in here we will search for any yaml/yml that could be a YOLO data file
!find /content/drive/MyDrive -iname "*.yaml" -o -iname "*.yml" | head -n 200

In [ ]:
#in here we will locate the YOLO frontside dataset folder on Drive and read its class names
!pip -q install pyyaml

from pathlib import Path
import yaml

# find the data.yaml for the dataset
matches = list(Path("/content/drive/MyDrive").rglob("data.yaml"))
print("Found data.yaml files:", len(matches))

# try to pick the one that lives inside "Egyptian IDs frontside" folder
frontside = None
for p in matches:
    if "frontside" in str(p).lower() and "egyptian" in str(p).lower():
        frontside = p
        break

print("Chosen data.yaml:", frontside)

DATASET_ROOT = frontside.parent if frontside else None
print("DATASET_ROOT:", DATASET_ROOT)

data = yaml.safe_load((DATASET_ROOT/"data.yaml").read_text())
print("names:", data.get("names"))
print("train:", data.get("train"))
print("val  :", data.get("val"))
print("test :", data.get("test"))

In [ ]:
!ls "/content/drive/MyDrive/Egyptian IDs frontside.v1i.yolov8"

In [ ]:
#in here we will fix data.yaml paths to match your folder structure (train/valid/test inside root)
from pathlib import Path
import yaml, shutil

DATASET_ROOT = Path("/content/drive/MyDrive/Egyptian IDs frontside.v1i 2.yolov8")
yaml_path = DATASET_ROOT / "data.yaml"

# backup
backup_path = DATASET_ROOT / "data_original.yaml"
shutil.copy(yaml_path, backup_path)
print("Backed up to:", backup_path)

data = yaml.safe_load(yaml_path.read_text())

data["train"] = "train/images"
data["val"]   = "valid/images"
data["test"]  = "test/images"

yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
print("Updated:", yaml_path)
print(yaml.safe_dump(data, sort_keys=False))

In [ ]:
#in here we will sanity-check that images/labels exist where YOLO expects them
from pathlib import Path

for split in ["train", "valid", "test"]:
    img_dir = DATASET_ROOT / split / "images"
    lab_dir = DATASET_ROOT / split / "labels"
    print(split, "| images:", img_dir.exists(), "| labels:", lab_dir.exists())
    if img_dir.exists():
        print("  sample imgs:", len(list(img_dir.glob("*"))) )
    if lab_dir.exists():
        print("  sample lbls:", len(list(lab_dir.glob("*.txt"))) )

In [ ]:
#in here we will train the YOLOv8 segmentation model for front-side field detection
!pip -q install ultralytics

from ultralytics import YOLO
from pathlib import Path

RUNS_DIR = Path("/content/drive/MyDrive/egyid_layer4_artifacts_front")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO("yolov8n-seg.pt")  # fast baseline; upgrade later if needed

results = model.train(
    data=str(DATASET_ROOT / "data.yaml"),
    imgsz=640,
    epochs=30,
    batch=16,
    device=0,
    project=str(RUNS_DIR),
    name="yolov8_front_fields_seg",
    exist_ok=True
)

In [ ]:
#in here we will load the trained Layer 4 front detector (best.pt) and verify it exists
from pathlib import Path
from ultralytics import YOLO

BEST_PT = Path("/content/drive/MyDrive/egyid_layer4_artifacts_front/yolov8_front_fields_seg/weights/best.pt")
print("BEST_PT exists:", BEST_PT.exists(), "| path:", BEST_PT)

detector = YOLO(str(BEST_PT))
print("Detector loaded ✅")

In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_PT = Path("/content/drive/MyDrive/egyid_layer4_artifacts_front/yolov8_front_fields_seg/weights/best.pt")

assert BEST_PT.exists(), f"best.pt not found at: {BEST_PT}"
detector = YOLO(str(BEST_PT))
print("Detector loaded ✅", BEST_PT)

In [40]:
# in here we will run Layer 4 detection on Layer 3 upright FRONT images and export crops + a manifest for OCR

import pandas as pd
import cv2
from pathlib import Path
from tqdm.notebook import tqdm
import yaml

# -----------------------
# 1️⃣ Auto-detect dataset root safely
# -----------------------
DATASET_ROOT = None
for p in Path("/content/drive/MyDrive").rglob("data.yaml"):
    if "frontside" in str(p).lower() and "egyptian" in str(p).lower():
        DATASET_ROOT = p.parent
        break

if DATASET_ROOT is None:
    raise RuntimeError("Could not auto-detect Egyptian frontside dataset. Check Drive.")

print("Using DATASET_ROOT:", DATASET_ROOT)

# -----------------------
# 2️⃣ Load class names
# -----------------------
data_yaml_path = DATASET_ROOT / "data.yaml"
if not data_yaml_path.exists():
    raise RuntimeError(f"data.yaml not found at {data_yaml_path}")

names = yaml.safe_load(data_yaml_path.read_text())["names"]
id2name = {i: n for i, n in enumerate(names)}
print("Classes:", id2name)

# -----------------------
# 3️⃣ Load Layer 3 manifest
# -----------------------
LAYER3_MANIFEST = Path("/content/drive/MyDrive/egyid_layer3_artifacts/layer3_manifest.csv")
if not LAYER3_MANIFEST.exists():
    raise RuntimeError("Layer 3 manifest not found.")

df3 = pd.read_csv(LAYER3_MANIFEST)

df_front = df3[
    (df3["status_layer3"] == "ok") &
    (df3["side"] == "front")
].copy()

print("Front images entering Layer 4:", len(df_front))

if len(df_front) == 0:
    raise RuntimeError("No front images found in Layer 3 manifest.")

# -----------------------
# 4️⃣ Ensure detector exists
# -----------------------
if "detector" not in globals():
    raise RuntimeError("Detector not loaded. Re-run YOLO loading cell first.")

# -----------------------
# 5️⃣ Run detection
# -----------------------
OUT_DIR = Path("/content/drive/MyDrive/egyid_layer4_artifacts_front")
CROPS_DIR = OUT_DIR / "crops"
CROPS_DIR.mkdir(parents=True, exist_ok=True)

rows = []

for _, r in tqdm(df_front.iterrows(), total=len(df_front)):
    img_path = r["upright_path"]

    if not Path(img_path).exists():
        rows.append({"status": "error", "reason": "missing_file", "upright_path": img_path})
        continue

    img = cv2.imread(img_path)
    if img is None:
        rows.append({"status": "error", "reason": "cannot_read", "upright_path": img_path})
        continue

    pred = detector.predict(source=img, imgsz=640, conf=0.25, verbose=False)[0]

    if pred.boxes is None or len(pred.boxes) == 0:
        rows.append({"status": "no_detections", "upright_path": img_path})
        continue

    h, w = img.shape[:2]

    for i in range(len(pred.boxes)):
        cls_id = int(pred.boxes.cls[i].item())
        conf  = float(pred.boxes.conf[i].item())
        x1, y1, x2, y2 = pred.boxes.xyxy[i].cpu().numpy().tolist()

        # clamp safely
        x1 = int(max(0, min(w-1, x1)))
        x2 = int(max(0, min(w, x2)))
        y1 = int(max(0, min(h-1, y1)))
        y2 = int(max(0, min(h, y2)))

        if x2 <= x1 or y2 <= y1:
            continue

        field = id2name.get(cls_id, str(cls_id))
        crop = img[y1:y2, x1:x2]

        crop_path = CROPS_DIR / f"{Path(img_path).stem}__{field}__{i}.jpg"
        cv2.imwrite(str(crop_path), crop)

        rows.append({
            "status": "ok",
            "upright_path": img_path,
            "field": field,
            "conf": conf,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "crop_path": str(crop_path)
        })

# -----------------------
# 6️⃣ Save results
# -----------------------
layer4_df = pd.DataFrame(rows)
LAYER4_MANIFEST = OUT_DIR / "layer4_front_detections.csv"
layer4_df.to_csv(LAYER4_MANIFEST, index=False)

print("\nSaved:", LAYER4_MANIFEST)
print(layer4_df["status"].value_counts(dropna=False))
layer4_df.head()

Using DATASET_ROOT: /content/drive/MyDrive/Egyptian IDs frontside.v1i 2.yolov8
Classes: {0: 'Address', 1: 'Birth', 2: 'ID', 3: 'Name'}
Front images entering Layer 4: 44


RuntimeError: Detector not loaded. Re-run YOLO loading cell first.

In [ ]:
#in here we will sanity-check crops by displaying one original and a few cropped fields
import matplotlib.pyplot as plt
import random

ok = layer4_df[layer4_df["status"]=="ok"]
sample_upright = ok["upright_path"].iloc[0]
print("Sample upright:", sample_upright)

img = cv2.cvtColor(cv2.imread(sample_upright), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8,5))
plt.imshow(img)
plt.axis("off")
plt.show()

sample_crops = ok[ok["upright_path"]==sample_upright].head(6)
for p in sample_crops["crop_path"].tolist():
    c = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6,2))
    plt.imshow(c)
    plt.title(Path(p).name)
    plt.axis("off")
    plt.show()

-----------


In [ ]:
#in here we will debug why there are no OK detections in layer4_df
print("Front images entering Layer 4:", len(df_front))
print(layer4_df["status"].value_counts(dropna=False))
layer4_df.head(20)

In [38]:
#in here we will sanity-check that Layer 3 upright paths exist and can be read
from pathlib import Path
import cv2

missing = 0
unreadable = 0

for p in df_front["upright_path"].head(20).tolist():
    if not Path(p).exists():
        missing += 1
        print("MISSING:", p)
        continue
    img = cv2.imread(p)
    if img is None:
        unreadable += 1
        print("UNREADABLE:", p)

print("Missing:", missing, "| Unreadable:", unreadable)
print("Example upright path:", df_front["upright_path"].iloc[0] if len(df_front) else "df_front is empty")

Missing: 0 | Unreadable: 0
Example upright path: /content/drive/MyDrive/egyid_layer3_artifacts/upright_images/front/ID177_rot0.jpg


In [39]:
#in here we will re-run Layer 4 inference using image paths (more reliable than passing cv2 arrays)
import pandas as pd
from pathlib import Path
from tqdm import tqdm

LAYER3_MANIFEST = Path("/content/drive/MyDrive/egyid_layer3_artifacts/layer3_manifest.csv")
df3 = pd.read_csv(LAYER3_MANIFEST)
df_front = df3[(df3["status_layer3"]=="ok") & (df3["side"]=="front")].copy()

print("Front images:", len(df_front))

test_rows = []
for p in df_front["upright_path"].head(10).tolist():
    pred = detector.predict(source=str(p), imgsz=640, conf=0.01, verbose=False)[0]
    n = 0 if pred.boxes is None else len(pred.boxes)
    mx = None if n == 0 else float(pred.boxes.conf.max().item())
    test_rows.append((Path(p).name, n, mx))

test_rows

Front images: 44


NameError: name 'detector' is not defined

In [36]:
# =========================
# Layer 5 (Front) = OCR Extraction
# What we do:
# 1) Load Layer 4 detections (crops + fields + confidence)
# 2) For each upright image, select the BEST crop per field (highest YOLO conf)
# 3) Run EasyOCR on each selected crop
# 4) Save results to CSV + JSON for downstream use
# =========================

import pandas as pd
from pathlib import Path
import os, json
import cv2

LAYER4_DIR = Path("/content/drive/MyDrive/egyid_layer4_artifacts_front")
LAYER4_MANIFEST = LAYER4_DIR / "layer4_front_detections.csv"

assert LAYER4_MANIFEST.exists(), f"Missing Layer4 manifest: {LAYER4_MANIFEST}"

df4 = pd.read_csv(LAYER4_MANIFEST)
print("Layer4 rows:", len(df4))
print(df4["status"].value_counts(dropna=False))

# Keep only successful detections
ok4 = df4[df4["status"] == "ok"].copy()
print("OK detections:", len(ok4))

# Optional: restrict to fields we care about
FIELDS = ["ID", "Name", "Birth", "Address"]  # change if you want
ok4 = ok4[ok4["field"].isin(FIELDS)].copy()
print("OK detections (selected fields):", len(ok4))
ok4.head()

Layer4 rows: 44
status
ok               22
no_detections    22
Name: count, dtype: int64
OK detections: 22
OK detections (selected fields): 22


,status,upright_path,field,conf,x1,y1,x2,y2,crop_path
0,ok,/content/drive/MyDrive/egyid_layer3_artifacts/...,ID,0.335954,615.0,639.0,864.0,687.0,/content/drive/MyDrive/egyid_layer4_artifacts_...
1,ok,/content/drive/MyDrive/egyid_layer3_artifacts/...,ID,0.286618,611.0,640.0,864.0,687.0,/content/drive/MyDrive/egyid_layer4_artifacts_...
2,ok,/content/drive/MyDrive/egyid_layer3_artifacts/...,ID,0.276513,614.0,640.0,861.0,686.0,/content/drive/MyDrive/egyid_layer4_artifacts_...
3,ok,/content/drive/MyDrive/egyid_layer3_artifacts/...,ID,0.292517,620.0,644.0,863.0,686.0,/content/drive/MyDrive/egyid_layer4_artifacts_...
4,ok,/content/drive/MyDrive/egyid_layer3_artifacts/...,ID,0.286625,617.0,641.0,864.0,687.0,/content/drive/MyDrive/egyid_layer4_artifacts_...


In [ ]:
# =========================
# Choose best crop per (upright_path, field) using highest YOLO confidence
# Output: one row per upright image per field (max 4 rows per image)
# =========================

# Make sure conf is numeric
ok4["conf"] = pd.to_numeric(ok4["conf"], errors="coerce")

best_crops = (
    ok4.sort_values(["upright_path", "field", "conf"], ascending=[True, True, False])
       .groupby(["upright_path", "field"], as_index=False)
       .head(1)
       .reset_index(drop=True)
)

print("Best crops rows:", len(best_crops))
best_crops.head(10)

In [ ]:
# =========================
# OCR engine setup
# NOTE: This may take 30-90s the first time (downloads models)
# =========================

!pip -q install easyocr

import torch
import easyocr

# Arabic + English is best for Egyptian IDs (Arabic names + digits)
reader = easyocr.Reader(['ar', 'en'], gpu=torch.cuda.is_available())
print("EasyOCR ready | GPU:", torch.cuda.is_available())

In [ ]:
# =========================
# Run OCR on each selected crop
# We return:
# - text (joined)
# - mean_conf
# - raw lines (optional)
# =========================

import numpy as np
from tqdm import tqdm

def ocr_crop(crop_path: str):
    img = cv2.imread(crop_path)
    if img is None:
        return {"status":"error", "reason":"cannot_read", "text":"", "mean_conf":0.0, "lines":[]}

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    try:
        res = reader.readtext(rgb, detail=1)
    except Exception as e:
        return {"status":"error", "reason":str(e), "text":"", "mean_conf":0.0, "lines":[]}

    if not res:
        return {"status":"empty", "reason":"no_text", "text":"", "mean_conf":0.0, "lines":[]}

    lines = []
    confs = []
    for (_bbox, txt, conf) in res:
        if isinstance(txt, str) and txt.strip():
            lines.append(txt.strip())
        if isinstance(conf, (float, int)):
            confs.append(float(conf))

    text = " ".join(lines).strip()
    mean_conf = float(np.mean(confs)) if confs else 0.0

    return {"status":"ok", "text":text, "mean_conf":mean_conf, "lines":lines}

rows = []
for _, r in tqdm(best_crops.iterrows(), total=len(best_crops)):
    crop_path = r["crop_path"]
    out = ocr_crop(crop_path)

    rows.append({
        "upright_path": r["upright_path"],
        "field": r["field"],
        "yolo_conf": float(r["conf"]) if pd.notna(r["conf"]) else None,
        "crop_path": crop_path,
        "ocr_status": out["status"],
        "ocr_text": out["text"],
        "ocr_mean_conf": out["mean_conf"],
        "ocr_lines": json.dumps(out["lines"], ensure_ascii=False),
    })

layer5_df = pd.DataFrame(rows)

OUT_DIR = Path("/content/drive/MyDrive/egyid_layer5_artifacts_front")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUT_DIR / "layer5_front_ocr.csv"
layer5_df.to_csv(CSV_PATH, index=False)
print("Saved:", CSV_PATH)

# Build per-image JSON
grouped = {}
for upright_path, g in layer5_df.groupby("upright_path"):
    obj = {"upright_path": upright_path}
    for _, rr in g.iterrows():
        obj[rr["field"]] = rr["ocr_text"]
    grouped[upright_path] = obj

JSON_PATH = OUT_DIR / "layer5_front_ocr.json"
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(list(grouped.values()), f, ensure_ascii=False, indent=2)

print("Saved:", JSON_PATH)

layer5_df.head(20)

In [ ]:
# =========================
# Quick check: print a few examples
# =========================
sample = layer5_df.sample(min(10, len(layer5_df)), random_state=42)
for _, r in sample.iterrows():
    print("\nFIELD:", r["field"])
    print("YOLO conf:", r["yolo_conf"], "| OCR conf:", r["ocr_mean_conf"])
    print("TEXT:", r["ocr_text"])

In [46]:
# =========================
# Layer 5 (Post-processing)
# Load Layer5 OCR CSV + normalize Arabic digits + extract Egyptian 14-digit National ID
# Output:
# - df5["ocr_text_clean"]
# - df5["id_14"]
# =========================

import pandas as pd
import re
from pathlib import Path

# ---- Load Layer 5 CSV ----
LAYER5_CSV = Path("/content/drive/MyDrive/egyid_layer5_artifacts_front/layer5_front_ocr.csv")
assert LAYER5_CSV.exists(), f"Layer5 CSV not found at: {LAYER5_CSV}"

df5 = pd.read_csv(LAYER5_CSV)
print("Layer5 rows:", len(df5))

# -------------------------
# Arabic → Western digit normalization
# -------------------------
AR2EN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize_digits(s):
    if not isinstance(s, str):
        return ""
    return s.translate(AR2EN)

df5["ocr_text_clean"] = df5["ocr_text"].fillna("").apply(normalize_digits)

# -------------------------
# Extract 14-digit national ID ONLY
# -------------------------
def extract_egypt_nid(text: str) -> str:
    """
    Extract Egyptian National ID (14 digits) from OCR text.
    Tries multiple patterns because OCR often outputs chunks like:
    '3722043 10 03 06 3'  (rest DD MM YY C)
    We rebuild it into:   C YY MM DD rest
    """
    if not isinstance(text, str):
        return ""

    # Get digit chunks as OCR produced them (keeps chunk boundaries)
    chunks = re.findall(r"\d+", text)
    if not chunks:
        return ""

    def valid(nid: str) -> bool:
        if len(nid) != 14:
            return False
        if nid[0] not in ("2", "3"):
            return False
        yy = int(nid[1:3])
        mm = int(nid[3:5])
        dd = int(nid[5:7])
        if not (1 <= mm <= 12):
            return False
        if not (1 <= dd <= 31):
            return False
        return True

    def try_windows(digits: str) -> str:
        digits = re.sub(r"\D", "", digits)
        if len(digits) < 14:
            return ""
        for i in range(0, len(digits) - 13):
            cand = digits[i:i+14]
            if valid(cand):
                return cand
        return ""

    # 1) Try normal concatenation windows
    digits_normal = "".join(chunks)
    hit = try_windows(digits_normal)
    if hit:
        return hit

    # 2) Common OCR patterns (reorder chunks into: C YY MM DD rest)
    lens = list(map(len, chunks))

    # Pattern A: [7][2][2][2][1] -> rest DD MM YY C
    if lens == [7, 2, 2, 2, 1]:
        cand = chunks[4] + chunks[3] + chunks[2] + chunks[1] + chunks[0]
        if valid(cand):
            return cand

    # Pattern B: [6][2][2][2][1]
    if lens == [6, 2, 2, 2, 1]:
        cand = chunks[4] + chunks[3] + chunks[2] + chunks[1] + chunks[0]
        if valid(cand):
            return cand

    # Pattern C: [2][1][7][2][1]
    if lens == [2, 1, 7, 2, 1]:
        rest_prefix, c, rest_main, dd, yy = chunks

        cand1 = c + yy + "01" + dd + rest_prefix + rest_main
        hit = try_windows(cand1)
        if hit:
            return hit

        cand2 = c + dd + yy + rest_prefix + rest_main
        hit = try_windows(cand2)
        if hit:
            return hit

    # Pattern D: [2][1][7][2]
    if lens == [2, 1, 7, 2]:
        rest_prefix, c, rest_main, dd = chunks
        cand = c + "00" + "01" + dd + rest_prefix + rest_main
        hit = try_windows(cand)
        if hit:
            return hit

    # Pattern E: [2][1][7][2][2]
    if lens == [2, 1, 7, 2, 2]:
        rest_prefix, c, rest_main, dd, mm_or_yy = chunks

        cand1 = c + mm_or_yy + "01" + dd + rest_prefix + rest_main
        hit = try_windows(cand1)
        if hit:
            return hit

        cand2 = c + "06" + mm_or_yy + dd + rest_prefix + rest_main
        hit = try_windows(cand2)
        if hit:
            return hit

    # 3) Try reversed concatenation windows
    digits_rev = "".join(chunks[::-1])
    hit = try_windows(digits_rev)
    if hit:
        return hit

    return ""

# Create id_14 column
df5["id_14"] = df5["ocr_text_clean"].apply(extract_egypt_nid)

# Summary
total_ids = (df5["field"].astype(str).str.lower() == "id").sum() if "field" in df5.columns else len(df5)
valid_ids = (df5["id_14"].astype(str).str.len() == 14).sum()

print(f"Valid IDs extracted: {valid_ids} out of {total_ids}")

# Preview
df5[df5["id_14"].astype(str).str.len() == 14][["ocr_text_clean", "id_14"]].head(10)

Layer5 rows: 22
Valid IDs extracted: 13 out of 22


,ocr_text_clean,id_14
0,3722043 10 03 06 3,30603103722043
1,1235728 03 01 08 3,30801031235728
2,4329371 24 04 87 2,28704244329371
4,9208579 11 03 97 2,29703119208579
5,77 2 6152397 15,20001157761523
8,70 2 2859102 10 06,20601107028591
9,2647387 27 1 1 96 2,29611272647387
11,80 2 4663136 21 08,20801218046631
12,99 2 8404476 24,20001249984044
15,6131308 31 08 93 2,29308316131308


In [47]:
# =========================
# Choose best candidate per field per card
# Priority = highest OCR confidence
# =========================

from pathlib import Path

# 1) Extract doc_id from upright_path (ID177 from ID177_rot0.jpg)
df5["doc_id"] = df5["upright_path"].apply(lambda p: Path(str(p)).stem.split("_")[0])

# 2) Ensure id_14 exists (because you only added the function, not the column)
if "id_14" not in df5.columns:
    df5["id_14"] = df5.apply(
        lambda r: extract_egypt_nid(r["ocr_text_clean"]) if r.get("field","") == "ID" else "",
        axis=1
    )

final_records = []

for doc_id, group in df5.groupby("doc_id"):
    record = {"doc_id": doc_id, "front": {}, "back": None}

    for field in ["ID", "Name", "Birth", "Address"]:
        g = group[group["field"] == field]
        if g.empty:
            continue

        # pick highest OCR confidence
        best_row = g.sort_values("ocr_mean_conf", ascending=False).iloc[0]

        if field == "ID":
            nid = best_row.get("id_14", "")
            if isinstance(nid, str) and nid.strip():
                record["front"]["id_number"] = nid.strip()
        else:
            record["front"][field.lower()] = str(best_row.get("ocr_text_clean", "")).strip()

    final_records.append(record)

print("Total structured records:", len(final_records))
final_records[:3]

Total structured records: 22


[{'doc_id': 'ID177', 'front': {'id_number': '30603103722043'}, 'back': None},
 {'doc_id': 'ID178', 'front': {'id_number': '30801031235728'}, 'back': None},
 {'doc_id': 'ID179', 'front': {'id_number': '28704244329371'}, 'back': None}]

In [48]:
OUT_DIR = Path("/content/drive/MyDrive/egyid_layer6_final")
OUT_DIR.mkdir(parents=True, exist_ok=True)

JSON_PATH = OUT_DIR / "layer6_front_final.json"

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(final_records, f, indent=2, ensure_ascii=False)

print("Saved final structured JSON to:")
print(JSON_PATH)

Saved final structured JSON to:
/content/drive/MyDrive/egyid_layer6_final/layer6_front_final.json


In [49]:
# =========================
# SAFE IMPROVEMENT: Retry OCR ONLY for missing ID_14 rows (no retraining)
# Requires:
#   - df5 already loaded (Layer5 CSV)
#   - reader already created (easyocr.Reader)
# =========================

import cv2
import numpy as np
import re
from pathlib import Path

# --- safety checks ---
assert "df5" in globals(), "df5 is not defined. Make sure you ran the cell that loads layer5_front_ocr.csv into df5."
assert "reader" in globals(), "reader is not defined. Make sure you ran the EasyOCR initialization cell: reader = easyocr.Reader([...])"

# --- Arabic -> Western digits ---
AR2EN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize_digits(s):
    if not isinstance(s, str):
        return ""
    return s.translate(AR2EN)

def valid_nid(nid: str) -> bool:
    if not isinstance(nid, str) or len(nid) != 14:
        return False
    if nid[0] not in ("2", "3"):
        return False
    yy = int(nid[1:3])
    mm = int(nid[3:5])
    dd = int(nid[5:7])
    return (1 <= mm <= 12) and (1 <= dd <= 31)

def extract_nid_from_text(text: str) -> str:
    # normalize arabic digits then keep only digits
    text = normalize_digits(text)
    digits = re.sub(r"\D", "", text)
    if len(digits) < 14:
        return ""
    # try all 14-digit windows
    for i in range(0, len(digits) - 13):
        cand = digits[i:i+14]
        if valid_nid(cand):
            return cand
    return ""

def ocr_text_and_conf(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    res = reader.readtext(rgb, detail=1)
    if not res:
        return "", 0.0
    texts = [r[1] for r in res if isinstance(r[1], str)]
    confs = [r[2] for r in res if isinstance(r[2], (float, int))]
    return " ".join(texts).strip(), float(np.mean(confs)) if confs else 0.0

def variants(img_bgr):
    # 1) original
    yield "orig", img_bgr

    # 2) upscaled
    up = cv2.resize(img_bgr, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    yield "up2x", up

    # 3) upscaled + adaptive threshold
    gray = cv2.cvtColor(up, cv2.COLOR_BGR2GRAY)
    thr = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY, 31, 5)
    thr_bgr = cv2.cvtColor(thr, cv2.COLOR_GRAY2BGR)
    yield "up2x_thr", thr_bgr


# --- retry only missing IDs ---
mask_missing = (df5["field"] == "ID") & (df5["id_14"].fillna("").astype(str).str.len() != 14)
missing_idx = df5[mask_missing].index.tolist()

print("Missing ID_14 rows to retry:", len(missing_idx))

fixed = 0
for idx in missing_idx:
    crop_path = df5.at[idx, "crop_path"]
    img = cv2.imread(str(crop_path))
    if img is None:
        continue

    best_nid = ""
    best_conf = -1.0
    best_txt  = ""

    for vname, vimg in variants(img):
        txt, conf = ocr_text_and_conf(vimg)
        nid = extract_nid_from_text(txt)

        # prefer valid NID; tie-break on confidence
        if nid and (best_nid == "" or conf > best_conf):
            best_nid, best_conf, best_txt = nid, conf, txt
        elif best_nid == "" and conf > best_conf:
            # keep best text even if not valid (for debugging)
            best_conf, best_txt = conf, txt

    if best_nid:
        df5.at[idx, "id_14"] = best_nid
        fixed += 1

print("Newly fixed IDs:", fixed)

# summary
total_ids = (df5["field"] == "ID").sum()
valid_now = (df5[df5["field"] == "ID"]["id_14"].fillna("").astype(str).str.len() == 14).sum()
print(f"Valid IDs extracted NOW: {valid_now} out of {total_ids}")

# Save improved CSV (new file, doesn't overwrite your original)
OUT_IMPROVED = Path("/content/drive/MyDrive/egyid_layer5_artifacts_front/layer5_front_ocr_improved.csv")
df5.to_csv(OUT_IMPROVED, index=False)
print("Saved improved CSV to:", OUT_IMPROVED)

Missing ID_14 rows to retry: 9


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Newly fixed IDs: 0
Valid IDs extracted NOW: 13 out of 22
Saved improved CSV to: /content/drive/MyDrive/egyid_layer5_artifacts_front/layer5_front_ocr_improved.csv


In [50]:
# =========================
# Layer 6 (Front Only) - Rebuild final JSON from improved Layer5 CSV
# =========================

import pandas as pd
import json
from pathlib import Path

LAYER5_IMPROVED = Path("/content/drive/MyDrive/egyid_layer5_artifacts_front/layer5_front_ocr_improved.csv")
assert LAYER5_IMPROVED.exists(), "Improved Layer5 CSV not found!"

df5 = pd.read_csv(LAYER5_IMPROVED)
print("Layer5 rows:", len(df5))

# Extract doc_id from upright_path (e.g., ID177 from ID177_rot0.jpg)
df5["doc_id"] = df5["upright_path"].apply(lambda p: Path(str(p)).stem.split("_")[0])

final_records = []
for doc_id, group in df5.groupby("doc_id"):
    record = {"doc_id": doc_id, "front": {}, "back": None}

    for field in ["ID", "Name", "Birth", "Address"]:
        g = group[group["field"] == field]
        if len(g) == 0:
            continue

        # best OCR candidate
        best_row = g.sort_values("ocr_mean_conf", ascending=False).iloc[0]

        if field == "ID":
            nid = str(best_row.get("id_14", "")).strip()
            if len(nid) == 14:
                record["front"]["id_number"] = nid
        else:
            record["front"][field.lower()] = str(best_row.get("ocr_text_clean", "")).strip()

    final_records.append(record)

print("Total structured records:", len(final_records))
print(final_records[:3])

OUT_DIR = Path("/content/drive/MyDrive/egyid_layer6_final")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSON = OUT_DIR / "layer6_front_final.json"

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(final_records, f, ensure_ascii=False, indent=2)

print("Saved final structured JSON to:")
print(OUT_JSON)

Layer5 rows: 22
Total structured records: 22
[{'doc_id': 'ID177', 'front': {}, 'back': None}, {'doc_id': 'ID178', 'front': {}, 'back': None}, {'doc_id': 'ID179', 'front': {}, 'back': None}]
Saved final structured JSON to:
/content/drive/MyDrive/egyid_layer6_final/layer6_front_final.json


In [52]:
# # =========================
# Layer 6 (Front Only) - ID only, very robust
# =========================

import pandas as pd
import json
import re
from pathlib import Path

LAYER5_IMPROVED = Path("/content/drive/MyDrive/egyid_layer5_artifacts_front/layer5_front_ocr_improved.csv")
assert LAYER5_IMPROVED.exists(), "Improved Layer5 CSV not found!"

df5 = pd.read_csv(LAYER5_IMPROVED, dtype=str)  # <--- IMPORTANT: keep everything as strings
print("Layer5 rows:", len(df5))

# safety: fill missing
for col in ["ocr_mean_conf", "ocr_text_clean", "id_14", "upright_path", "field"]:
    if col not in df5.columns:
        df5[col] = ""
    df5[col] = df5[col].fillna("").astype(str)

# numeric conf for sorting
df5["ocr_mean_conf_num"] = pd.to_numeric(df5["ocr_mean_conf"], errors="coerce").fillna(0.0)

# normalize field + doc_id
df5["field_norm"] = df5["field"].str.strip().str.upper()
df5["doc_id"] = df5["upright_path"].apply(lambda p: Path(str(p)).stem.split("_")[0])

def find_14_digits(x: str) -> str:
    """Return first 14-digit sequence found in a messy string, else ''."""
    x = str(x)
    m = re.search(r"\d{14}", x)
    return m.group(0) if m else ""

final_records = []
for doc_id, group in df5.groupby("doc_id"):
    record = {"doc_id": doc_id, "front": {}, "back": None}

    g_id = group[group["field_norm"] == "ID"]
    if len(g_id):
        best = g_id.sort_values("ocr_mean_conf_num", ascending=False).iloc[0]

        nid = find_14_digits(best["id_14"])
        # fallback: if id_14 column is empty/messed, try extracting from cleaned OCR text
        if not nid:
            nid = find_14_digits(best["ocr_text_clean"])

        if nid:
            record["front"]["id_number"] = nid

    final_records.append(record)

print("Total structured records:", len(final_records))
print("Records with an extracted ID:",
      sum(1 for r in final_records if "id_number" in r["front"]))
print(final_records[:3])

OUT_DIR = Path("/content/drive/MyDrive/egyid_layer6_final")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSON = OUT_DIR / "layer6_front_final.json"

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(final_records, f, ensure_ascii=False, indent=2)

print("Saved final structured JSON to:")
print(OUT_JSON)

Layer5 rows: 22
Total structured records: 22
Records with an extracted ID: 13
[{'doc_id': 'ID177', 'front': {'id_number': '30603103722043'}, 'back': None}, {'doc_id': 'ID178', 'front': {'id_number': '30801031235728'}, 'back': None}, {'doc_id': 'ID179', 'front': {'id_number': '28704244329371'}, 'back': None}]
Saved final structured JSON to:
/content/drive/MyDrive/egyid_layer6_final/layer6_front_final.json
